[![Open In Colab](https://google.com)](https://colab.research.google.com/github/hamagami/IS2026/blob/main/Demo_27_EpsilonGreedy_Bandit.ipynb)

# Demo 27: Exploration and Exploitation: ε-Greedy / 探索と活用

Explore how ε changes the balance between the best-known action and random trials in a three-armed bandit. / 3本のバンディット問題で、最善と推定した行動とランダム探索の比率を変えて観察する。

**How to use / 使い方：** Run the cells from top to bottom, then change the controls and compare the figures. / セルを上から順に実行し、操作項目を変えて図を比較する。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass
import ipywidgets as widgets
plt.rcParams.update({'figure.dpi': 105, 'font.size': 11})
np.set_printoptions(precision=3, suppress=True)

In [ ]:
ARM_NAMES=['A','B','C'];TRUE_MEANS=np.array([0.2,0.5,0.8])
def bandit_run(epsilon=.1,n=1000,seed=0):
    rng=np.random.default_rng(seed);q=np.zeros(3);cnt=np.zeros(3,dtype=int)
    actions=[];rewards=[];est=[];regret=[]
    for t in range(n):
        if rng.random()<epsilon:a=int(rng.integers(3))
        else:a=int(rng.choice(np.flatnonzero(np.isclose(q,q.max()))))
        r=float(rng.normal(TRUE_MEANS[a],.6));cnt[a]+=1;q[a]+=(r-q[a])/cnt[a]
        actions.append(a);rewards.append(r);est.append(q.copy());regret.append(TRUE_MEANS.max()-TRUE_MEANS[a])
    return np.array(actions),np.array(rewards),np.array(est),np.cumsum(regret)

def batch_bandits(eps_list=(0,.1,.5,1),horizon=1000,repeats=50,seed=0):
    data={}
    for e in eps_list:
        rg=np.random.default_rng(seed+int(e*1000));curves=[]
        for _ in range(repeats):
            a,r,q,reg=bandit_run(e,horizon,int(rg.integers(2**30)))
            curves.append(np.cumsum(r)/np.arange(1,horizon+1))
        data[e]=np.mean(curves,axis=0)
    return data

In [ ]:
epsilon_w=widgets.FloatSlider(value=.1,min=0,max=1,step=.05,description='ε',continuous_update=False)
trials=widgets.IntSlider(value=1000,min=200,max=3000,step=200,description='Trials / 試行',continuous_update=False)
seed=widgets.IntSlider(value=10,min=0,max=100,description='Seed / 乱数')
button=widgets.Button(description='Run / 実行',button_style='primary');out=widgets.Output()
def update(_):
    with out:
        clear_output(wait=True)
        act,rew,q,reg=bandit_run(epsilon_w.value,trials.value,seed.value)
        fig,ax=plt.subplots(1,3,figsize=(14,3.7))
        ax[0].bar(ARM_NAMES,np.bincount(act,minlength=3));ax[0].set(title='Chosen actions / 選択行動',ylabel='Count / 回数')
        ax[1].plot(q);ax[1].axhline(TRUE_MEANS.max(),ls=':',c='k',alpha=.6,label='Best mean / 最良平均')
        ax[1].set(title='Estimated Q / 推定Q値',xlabel='Trial / 試行',ylabel='Estimate / 推定平均');ax[1].legend()
        ax[2].plot(reg);ax[2].set(title='Cumulative regret / 累積後悔',xlabel='Trial / 試行',ylabel='Regret / 後悔')
        for a in ax:a.grid(alpha=.2)
        plt.tight_layout();plt.show()
        print('True mean rewards / 真の平均報酬（Agentには未知）:',dict(zip(ARM_NAMES,TRUE_MEANS)))
        print('Estimated means / 推定平均:',np.round(q[-1],3),'| best-arm rate / 最適選択率:',round(np.mean(act==2),3))
        avg=batch_bandits(horizon=trials.value,repeats=30,seed=seed.value)
        plt.figure(figsize=(9,3.5))
        for e,curve in avg.items():plt.plot(curve,label=f'ε = {e}')
        plt.title('Mean reward (30 runs) / 平均報酬（30回）');plt.xlabel('Trials / 試行');plt.ylabel('Mean reward / 平均報酬');plt.legend();plt.grid(alpha=.2);plt.show()
button.on_click(update)
display(widgets.VBox([widgets.HBox([epsilon_w,trials,seed]),button,out]));update(None)

## ε-Greedy action probabilities / ε-greedyの行動選択確率

With probability $1-\varepsilon$, select an action currently estimated to be best; with probability $\varepsilon$, select **uniformly among all actions**. The random choice may also select the best action. / 確率 $1-\varepsilon$ で現在最善と推定した行動を選び、確率 $\varepsilon$ で **すべての行動から一様に** 選ぶ。ランダム選択でも最善の行動が選ばれうる。

**Safety:** this is a simulated bandit. Unrestricted random torques on real robots may be dangerous. / **安全性：** このデモは仮想バンディットである。実機で関節トルクを無制限にランダム探索することは危険になりうる。

## Try it yourself / 実験課題

1. Find a seed where ε=0 gets stuck on a suboptimal action. / ε=0で最適でない行動に固定される乱数種を探す。
2. Compare average rewards for ε=0.1 and ε=1.0. / ε=0.1と1.0の平均報酬を比較する。
3. Explain why the probability of choosing the best action can exceed $1-arepsilon$. / 最善行動が選ばれる確率が $1-arepsilon$ より大きくなりうる理由を説明する。